# E-17: two-sided bounds under misspecification (D type, H17)

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-17, §1.9 and Amendment 3. Verifies `prop:misspecified_projection` on 288 finite-support cells: 24 propensity designs x 2 spans x 6 generators.

- Non-SQ cells whose population minimizer is certified interior (§1.9 on the support points) and whose dual coordinate is misspecified: both bounds (line and range versions of the curvature constants) at relative tolerance 1e-10, the orthogonality and the Riesz weighted bias identity at 1e-12, and the bias bound.
- The 16 correctly specified UKL cells (Amendment 3): ||e*|| <= 1e-10, orthogonality and identity at 1e-12.
- SQ cells are reported separately: mu = C_g = 2 and equality in both bounds.

Run only through `python3 notebooks/experiments/run_registered.py E-17 stage0 15_E17_misspecification_bounds.ipynb`. A solver that raises stops the notebook, and the run is recorded as failed (registration §1.1).

In [ ]:
import itertools
import warnings

import matplotlib
import numpy as np
import pandas as pd

from grrexp import env, outputs, population

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

gr = env.use_submodule_genriesz()
rec = outputs.RunRecorder(17, "stage0")

REL, ABS_ID, ABS_SPEC = 1e-10, 1e-12, 1e-10
SPURIOUS_MATMUL = r"(?:divide by zero|overflow|invalid value) encountered in matmul$"

Z = np.linspace(-1.0, 1.0, 21)
DESIGNS = list(itertools.product((0.0, 0.5), (0.5, 1.5), (0.0, 1.0, 2.0), (0.0, 1.0)))  # a0, a1, a2, a3
SPANS = {"lin": 1, "quad": 2}


def sign(A):
    return np.where(np.atleast_2d(A)[:, 0] == 1, 1, -1)


GENERATORS = {
    "SQ": lambda: gr.SquaredGenerator(C=0.0),
    "UKL(1)": lambda: gr.UKLGenerator(C=1.0, branch_fn=sign),
    "BKL(1)": lambda: gr.BKLGenerator(C=1.0, branch_fn=sign),
    "BP(0.25,0)": lambda: gr.BPGenerator(omega=0.25, C=0.0, branch_fn=sign),
    "BP(0.5,0)": lambda: gr.BPGenerator(omega=0.5, C=0.0, branch_fn=sign),
    "BP(0.75,0)": lambda: gr.BPGenerator(omega=0.75, C=0.0, branch_fn=sign),
}
assert len(DESIGNS) * len(SPANS) * len(GENERATORS) == 288


def correctly_specified(gen_name, span, a2, a3):
    # Amendment 3: the UKL(1) dual coordinate of alpha0 is -(a0 + a1 z + a2 z^2 + a3 sin(pi z)) on both arms
    return gen_name == "UKL(1)" and a3 == 0.0 and (SPANS[span] == 2 or a2 == 0.0)

In [ ]:
def population_of(a0, a1, a2, a3):
    e = 1.0 / (1.0 + np.exp(-(a0 + a1 * Z + a2 * Z**2 + a3 * np.sin(np.pi * Z))))
    X = np.array([(d, z) for z in Z for d in (1.0, 0.0)])
    ez = np.repeat(e, 2)
    w = np.where(X[:, 0] == 1, ez, 1.0 - ez) / Z.size
    alpha0 = np.where(X[:, 0] == 1, 1.0 / ez, -1.0 / (1.0 - ez))
    gamma0 = np.where(X[:, 0] == 1, X[:, 1] ** 2 + np.sin(np.pi * X[:, 1]), 0.5 * X[:, 1])
    theta0 = float(np.mean(Z**2 + np.sin(np.pi * Z) - 0.5 * Z))
    return X, w, alpha0, gamma0, theta0


def design_matrices(X, deg):
    base = gr.CallableBasis(lambda A: np.column_stack([np.atleast_2d(A)[:, 0] ** k for k in range(deg + 1)]))
    basis = gr.TreatmentInteractionBasis(base_basis=base)
    basis.fit(X)
    fn = gr.ATEFunctional(0)
    Xc = np.asarray(fn.evaluation_points(X, basis))
    return np.asarray(basis(X)), np.asarray(fn.m_basis_matrix(X, basis)), Xc, np.asarray(basis(Xc))


def g2(gen, X, a):
    # g''(a) = 1 / (d alpha / d v) at v = g'(a), through the generator's own link
    v = np.asarray(gen.grad(X, a), float)
    return 1.0 / np.asarray(gen.dual_eval(X, v)[2], float)


def curvature(gen, X, a_star, a0):
    # line version: g'' on each segment [alpha*(x), alpha0(x)]; every registered g'' is monotone in |alpha| on a
    # branch, so the extremes are at the end points (checked on a grid of 201 points per segment)
    lo_end = np.minimum(g2(gen, X, a_star), g2(gen, X, a0))
    hi_end = np.maximum(g2(gen, X, a_star), g2(gen, X, a0))
    grid = np.linspace(0.0, 1.0, 201)
    for i in range(len(X)):
        seg = a_star[i] + grid * (a0[i] - a_star[i])
        vals = g2(gen, np.repeat(X[i : i + 1], grid.size, axis=0), seg)
        assert vals.min() >= lo_end[i] * (1 - 1e-12) and vals.max() <= hi_end[i] * (1 + 1e-12), "g'' not monotone"
    mu_line, C_line = float(lo_end.min()), float(hi_end.max())
    # range version: the convex hull of {alpha*(x), alpha0(x)} on each sign component
    mus, Cs = [], []
    for s in (1, -1):
        rows = np.where(sign(X) == s)[0]
        vals = np.concatenate([a_star[rows], a0[rows]])
        ends = np.array([vals.min(), vals.max()])
        Xs = np.repeat(X[rows[:1]], 2, axis=0)
        gg = g2(gen, Xs, ends)
        mus.append(gg.min())
        Cs.append(gg.max())
    return mu_line, C_line, float(min(mus)), float(max(Cs))

In [ ]:
records = []
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", message=SPURIOUS_MATMUL, category=RuntimeWarning)
    for (a0, a1, a2, a3), (span, deg), gen_name in itertools.product(DESIGNS, SPANS.items(), GENERATORS):
        gen = GENERATORS[gen_name]()
        X, w, alpha0, gamma0, theta0 = population_of(a0, a1, a2, a3)
        Phi, M, Xc, Phic = design_matrices(X, deg)
        ref = np.where(sign(X) == 1, 2.0, -2.0)
        u_ref = np.asarray(gen.grad(X, ref), float)
        u_ref_c = np.asarray(gen.grad(Xc, np.where(sign(Xc) == 1, 2.0, -2.0)), float)
        sol = population.solve(generator=gen, X=X, w=w, Phi=Phi, M=M, offset=u_ref,
                               check_X=Xc, check_Phi=Phic, check_offset=u_ref_c)
        a_star = sol.alpha
        e = a_star - alpha0
        r = (np.asarray(gen.grad(X, alpha0), float) - u_ref)
        delta = population.l2_norm(r - population.weighted_projection(r, Phi, w), w)
        norm_e = population.l2_norm(e, w)
        res_gamma = gamma0 - population.weighted_projection(gamma0, Phi, w)
        mu_l, C_l, mu_r, C_r = curvature(gen, X, a_star, alpha0)
        bias = float(w @ (a_star * gamma0)) - theta0
        records.append({
            "a0": a0, "a1": a1, "a2": a2, "a3": a3, "span": span, "generator": gen_name,
            "status": sol.status, "certified": sol.certified(), "max_gradient": sol.max_gradient,
            "min_hessian_eig": sol.min_hessian_eig, "min_margin": sol.min_margin,
            "correctly_specified": correctly_specified(gen_name, span, a2, a3),
            "delta": delta, "norm_e": norm_e,
            "mu_line": mu_l, "C_line": C_l, "mu_range": mu_r, "C_range": C_r,
            "orthogonality": float(np.max(np.abs(Phi.T @ (w * e)))),
            "bias": bias, "identity_gap": abs(bias - float(w @ (e * res_gamma))),
            "res_gamma_norm": population.l2_norm(res_gamma, w),
        })
cells_df = pd.DataFrame(records)
warning_messages = [str(x.message) for x in caught]
len(cells_df), warning_messages[:5]

In [ ]:
df = cells_df.copy()
df["sq"] = df["generator"] == "SQ"
for v in ("line", "range"):
    df[f"lower_ok_{v}"] = df["delta"] / df[f"C_{v}"] <= df["norm_e"] * (1 + REL)
    df[f"upper_ok_{v}"] = df["norm_e"] <= df["delta"] / df[f"mu_{v}"] * (1 + REL)
df["orth_ok"] = df["orthogonality"] <= ABS_ID
df["identity_ok"] = df["identity_gap"] <= ABS_ID
df["bias_bound_ok"] = df["bias"].abs() <= df["delta"] / df["mu_line"] * df["res_gamma_norm"] * (1 + REL) + ABS_ID
df["spec_ok"] = df["norm_e"] <= ABS_SPEC

judged = df[df["certified"] & ~df["sq"] & ~df["correctly_specified"]]
bound_cols = ["lower_ok_line", "upper_ok_line", "lower_ok_range", "upper_ok_range",
              "orth_ok", "identity_ok", "bias_bound_ok"]
violations_main = judged[~judged[bound_cols].all(axis=1)]
spec = df[df["correctly_specified"]]
spec_judged = spec[spec["certified"]]
violations_spec = spec_judged[~spec_judged[["spec_ok", "orth_ok", "identity_ok"]].all(axis=1)]
sq = df[df["sq"]]
sq_curv = bool(np.allclose(sq[["mu_line", "C_line", "mu_range", "C_range"]].to_numpy(), 2.0, rtol=0, atol=1e-12))
sq_equality = (sq["norm_e"] - sq["delta"] / 2.0).abs() <= REL * np.maximum(1.0, sq["delta"] / 2.0)
unexpected_zero = df[(df["delta"] <= 1e-12) & ~df["correctly_specified"]]
not_certified = df[~df["certified"]]

summary = {
    "cells": int(len(df)),
    "judged_non_sq_interior": int(len(judged)),
    "violations_non_sq": int(len(violations_main)),
    "correctly_specified_cells": int(len(spec)),
    "correctly_specified_certified": int(len(spec_judged)),
    "violations_correctly_specified": int(len(violations_spec)),
    "sq_cells": int(len(sq)),
    "sq_curvature_two": sq_curv,
    "sq_equality_failures": int((~sq_equality).sum()),
    "sq_orthogonality_failures": int((~sq["orth_ok"]).sum()),
    "not_certified": int(len(not_certified)),
    "unexpected_delta_zero": int(len(unexpected_zero)),
}
negative = (summary["violations_non_sq"] > 0 or summary["violations_correctly_specified"] > 0
            or not sq_curv or summary["sq_equality_failures"] > 0)
print(summary, "negative result:", negative)
rec.write_stage0("e17_cells", {
    "summary": summary,
    "negative_result": bool(negative),
    "cells": [
        {k: (None if isinstance(v, float) and not np.isfinite(v) else (bool(v) if isinstance(v, np.bool_) else v))
         for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ],
    "warnings": warning_messages,
})

In [ ]:
def tex_row(cells):
    return " & ".join(str(c) for c in cells) + r" \\"


lines = [r"\begin{tabular}{llrrrr}", r"\hline",
         tex_row(["Generator", "Span", "Cells", "Interior", "Correctly specified", "Violations"]), r"\hline"]
for gen_name in GENERATORS:
    if gen_name == "SQ":
        continue
    for span in SPANS:
        sub = df[(df["generator"] == gen_name) & (df["span"] == span)]
        js = sub[sub["certified"] & ~sub["correctly_specified"]]
        ss = sub[sub["certified"] & sub["correctly_specified"]]
        nviol = int((~js[bound_cols].all(axis=1)).sum()) + int((~ss[["spec_ok", "orth_ok", "identity_ok"]].all(axis=1)).sum())
        lines.append(tex_row([gen_name, span, len(sub), int(sub["certified"].sum()), int(sub["correctly_specified"].sum()), nviol]))
lines += [r"\hline"]
for span in SPANS:
    sub = df[(df["generator"] == "SQ") & (df["span"] == span)]
    eq_ok = int(((sub["norm_e"] - sub["delta"] / 2.0).abs() <= REL * np.maximum(1.0, sub["delta"] / 2.0)).sum())
    lines.append(tex_row(["SQ (equality)", span, len(sub), int(sub["certified"].sum()), 0, len(sub) - eq_ok]))
lines += [r"\hline", r"\end{tabular}"]
rec.write_table("tab_E17", "\n".join(lines) + "\n")

plot = judged[judged["delta"] > 0].copy()
plot["t"] = (plot["norm_e"] - plot["delta"] / plot["C_line"]) / (plot["delta"] / plot["mu_line"] - plot["delta"] / plot["C_line"])
fig, ax = plt.subplots(figsize=(6.0, 3.2))
names = [g for g in GENERATORS if g != "SQ"]
for k, gen_name in enumerate(names):
    for j, span in enumerate(SPANS):
        sub = plot[(plot["generator"] == gen_name) & (plot["span"] == span)]
        ax.scatter(sub["t"], np.full(len(sub), k + (j - 0.5) * 0.25), s=10, marker="o" if span == "lin" else "s",
                   label=f"span {span}" if k == 0 else None)
ax.axvline(0.0, color="0.4", lw=0.8)
ax.axvline(1.0, color="0.4", lw=0.8)
ax.set_yticks(range(len(names)), names)
ax.set_xlabel(r"normalized position $t$ of $\|\alpha^*-\alpha_0\|$ between the lower and upper bounds")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
rec.save_figure(fig, "fig_E17_bounds")
rec.finalize(R=None, n=None, warnings=len(warning_messages),
             failures=summary["violations_non_sq"] + summary["violations_correctly_specified"] + summary["sq_equality_failures"])